# 03 — Machine Learning + Evaluación
**Credit Risk Analyzer** | German Credit Dataset (UCI)

**Integrante 2:** Machine Learning + Evaluación

**Inputs (del Integrante 1):**
- `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`
- `data/processed/y_train.csv`, `y_val.csv`, `y_test.csv`
- `data/processed/scaler.pkl`, `ordinal_encoder.pkl`, `feature_columns.json`

**Outputs de este notebook:**
- `models/checkpoints/best_model.pkl`
- `models/checkpoints/model_metadata.json`
- `models/plots/*.png`

In [ ]:
# Instalar dependencias si es necesario
# (correr solo una vez, luego comentar)
# !pip install xgboost scikit-learn pandas numpy matplotlib joblib

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay,
    precision_recall_curve, average_precision_score
)
from xgboost import XGBClassifier

os.makedirs('../models/checkpoints', exist_ok=True)
os.makedirs('../models/plots', exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Librerías cargadas ✓')

## 1. Cargar Splits del Integrante 1

> El preprocessing ya está hecho. NO se vuelve a hacer split ni encoding aquí.

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val   = pd.read_csv('../data/processed/X_val.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val   = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

with open('../data/processed/feature_columns.json') as f:
    FEATURE_NAMES = json.load(f)

print(f'Train : {X_train.shape[0]} filas | {X_train.shape[1]} features')
print(f'Val   : {X_val.shape[0]} filas')
print(f'Test  : {X_test.shape[0]} filas')
print(f'\nFeatures ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')
print(f'\nDistribución clases en train:')
print(f'  0 (good): {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')
print(f'  1 (bad):  {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print('\n⚠️  Desbalance 70/30 — todos los modelos usan class_weight=balanced')

## 2. Baseline — Logistic Regression

> Punto de referencia mínimo. Cualquier modelo que entrenemos debe superar este baseline.

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_val)
baseline_proba = baseline_model.predict_proba(X_val)[:, 1]

print('=== BASELINE: Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_val, baseline_preds):.4f}')
print(f'Precision: {precision_score(y_val, baseline_preds):.4f}')
print(f'Recall   : {recall_score(y_val, baseline_preds):.4f}')
print(f'F1       : {f1_score(y_val, baseline_preds):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_val, baseline_proba):.4f}')
print()
print(classification_report(y_val, baseline_preds, target_names=['good (0)', 'bad (1)']))

BASELINE_ROC_AUC = roc_auc_score(y_val, baseline_proba)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, baseline_preds,
                                         display_labels=['good', 'bad'], ax=ax)
ax.set_title('Confusion Matrix — Baseline LR')
plt.tight_layout()
plt.savefig('../models/plots/cm_baseline.png', dpi=150)
plt.show()

## 3. Modelos Principales

> Random Forest y XGBoost. Ambos con manejo de desbalance incorporado.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=SEED
)
rf_model.fit(X_train, y_train)
print('Random Forest entrenado ✓')

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=2.33,   # ratio 70/30 → penaliza más los errores en clase 'bad'
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED
)
xgb_model.fit(X_train, y_train)
print('XGBoost entrenado ✓')

## 4. Comparación de Modelos

In [ ]:
def get_metrics(model, X, y):
    preds = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y, preds), 4),
        'Precision': round(precision_score(y, preds), 4),
        'Recall':    round(recall_score(y, preds), 4),
        'F1':        round(f1_score(y, preds), 4),
        'ROC-AUC':   round(roc_auc_score(y, proba), 4),
    }

# Cross-validation ROC-AUC sobre train
def cv_roc_auc(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    return scores.mean(), scores.std()

print('Calculando CV scores (puede tardar ~1 min)...')
baseline_cv = cv_roc_auc(baseline_model, X_train, y_train)
rf_cv       = cv_roc_auc(rf_model,       X_train, y_train)
xgb_cv      = cv_roc_auc(xgb_model,      X_train, y_train)

print(f'Baseline LR  CV ROC-AUC: {baseline_cv[0]:.4f} ± {baseline_cv[1]:.4f}')
print(f'RandomForest CV ROC-AUC: {rf_cv[0]:.4f} ± {rf_cv[1]:.4f}')
print(f'XGBoost      CV ROC-AUC: {xgb_cv[0]:.4f} ± {xgb_cv[1]:.4f}')

In [ ]:
results = pd.DataFrame({
    'Model':      ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'CV ROC-AUC': [
        f"{baseline_cv[0]:.4f} ± {baseline_cv[1]:.4f}",
        f"{rf_cv[0]:.4f} ± {rf_cv[1]:.4f}",
        f"{xgb_cv[0]:.4f} ± {xgb_cv[1]:.4f}",
    ],
    **{k: [v] for k, v in get_metrics(baseline_model, X_val, y_val).items()},
})

# Reconstruir correctamente fila por fila
rows = []
for name, model, cv in [
    ('Logistic Regression', baseline_model, baseline_cv),
    ('Random Forest',       rf_model,       rf_cv),
    ('XGBoost',             xgb_model,      xgb_cv),
]:
    m = get_metrics(model, X_val, y_val)
    rows.append({'Model': name, 'CV ROC-AUC': f"{cv[0]:.4f} ± {cv[1]:.4f}", **m})

results = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print('=== Comparación en Val Set ===')
results

In [ ]:
# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 4))
models_names = ['Baseline LR', 'Random Forest', 'XGBoost']
val_rocs = [
    roc_auc_score(y_val, baseline_model.predict_proba(X_val)[:, 1]),
    roc_auc_score(y_val, rf_model.predict_proba(X_val)[:, 1]),
    roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1]),
]
cv_means = [baseline_cv[0], rf_cv[0], xgb_cv[0]]
cv_stds  = [baseline_cv[1], rf_cv[1], xgb_cv[1]]

x = np.arange(len(models_names))
ax.bar(x - 0.2, cv_means, 0.35, yerr=cv_stds, label='CV ROC-AUC',  color='#457b9d', capsize=4)
ax.bar(x + 0.2, val_rocs, 0.35,               label='Val ROC-AUC', color='#e63946')
ax.set_xticks(x)
ax.set_xticklabels(models_names)
ax.set_ylabel('ROC-AUC')
ax.set_title('Comparación de Modelos — German Credit')
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('../models/plots/model_comparison.png', dpi=150)
plt.show()

## 5. Seleccionar Mejor Modelo y Evaluar en Test Set

> Test set = datos **nunca vistos** durante entrenamiento ni selección de modelo.

In [ ]:
# Seleccionar el de mayor Val ROC-AUC
best_val_rocs = {
    'logistic_regression': val_rocs[0],
    'random_forest':       val_rocs[1],
    'xgboost':             val_rocs[2],
}
best_name  = max(best_val_rocs, key=best_val_rocs.get)
best_model = {'logistic_regression': baseline_model,
              'random_forest': rf_model,
              'xgboost': xgb_model}[best_name]

print(f'Mejor modelo: {best_name} (Val ROC-AUC = {best_val_rocs[best_name]:.4f})')

test_preds = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    'accuracy':  round(accuracy_score(y_test, test_preds), 4),
    'precision': round(precision_score(y_test, test_preds), 4),
    'recall':    round(recall_score(y_test, test_preds), 4),
    'f1':        round(f1_score(y_test, test_preds), 4),
    'roc_auc':   round(roc_auc_score(y_test, test_proba), 4),
}

print('\n=== EVALUACIÓN FINAL EN TEST SET ===')
for k, v in test_metrics.items():
    print(f'  {k:<10}: {v}')
print()
print(classification_report(y_test, test_preds, target_names=['good (0)', 'bad (1)']))

## 6. Visualizaciones del Mejor Modelo

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_preds,
    display_labels=['good', 'bad'], ax=ax
)
ax.set_title(f'Confusion Matrix — {best_name} [TEST]')
plt.tight_layout()
plt.savefig(f'../models/plots/cm_{best_name}.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, test_proba, name=best_name, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_title(f'ROC Curve — {best_name} [TEST]')
ax.legend()
plt.tight_layout()
plt.savefig(f'../models/plots/roc_{best_name}.png', dpi=150)
plt.show()

In [ ]:
# Precision-Recall Curve — más informativa que ROC en datasets desbalanceados
precision_vals, recall_vals, _ = precision_recall_curve(y_test, test_proba)
ap = average_precision_score(y_test, test_proba)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(recall_vals, precision_vals, label=f'AP = {ap:.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall — {best_name} [TEST]')
ax.legend()
plt.tight_layout()
plt.savefig(f'../models/plots/pr_{best_name}.png', dpi=150)
plt.show()

## 7. Feature Importance

> Según el EDA del Integrante 1 deberían aparecer arriba:
> `Credit amount`, `Duration`, `Checking account`, `Saving accounts`.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    indices = np.argsort(importance)[::-1][:15]
    top_features = [FEATURE_NAMES[i] for i in indices]
    top_importance = importance[indices]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(top_features[::-1], top_importance[::-1], color='#457b9d')
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_xlabel('Importancia')
    ax.set_title(f'Feature Importance — {best_name} (Top 15)')
    plt.tight_layout()
    plt.savefig(f'../models/plots/fi_{best_name}.png', dpi=150)
    plt.show()

    importance_df = pd.DataFrame({
        'feature': FEATURE_NAMES,
        'importance': importance
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    print('Top 10 features:')
    print(importance_df.head(10).to_string(index=False))
else:
    print(f'{best_name} no tiene feature_importances_. Usar coeficientes:')
    coef_df = pd.DataFrame({
        'feature': FEATURE_NAMES,
        'coef': np.abs(best_model.coef_[0])
    }).sort_values('coef', ascending=False)
    print(coef_df.head(10).to_string(index=False))
    importance_df = coef_df.rename(columns={'coef': 'importance'})

## 8. Guardar Modelo + Metadata

In [ ]:
# Guardar como best_model.pkl (el Integrante 3 lo carga así)
joblib.dump(best_model, '../models/checkpoints/best_model.pkl')
print('Guardado: models/checkpoints/best_model.pkl')

metadata = {
    'best_model_name': best_name,
    'feature_names': FEATURE_NAMES,
    'target_col': 'Risk',
    'target_mapping': {'0': 'good (bajo riesgo)', '1': 'bad (alto riesgo)'},
    'n_classes': 2,
    'dataset': 'German Credit Risk (UCI)',
    'class_distribution': {'good': '70%', 'bad': '30%'},
    'imbalance_strategy': 'class_weight=balanced / scale_pos_weight=2.33',
    'risk_thresholds': {'bajo': '<30%', 'medio': '30-55%', 'alto': '>55%'},
    'preprocessing_artifacts': {
        'scaler':          'data/processed/scaler.pkl',
        'ordinal_encoder': 'data/processed/ordinal_encoder.pkl',
        'feature_columns': 'data/processed/feature_columns.json'
    },
    'test_metrics': test_metrics,
    'top_features': importance_df.head(5)['feature'].tolist(),
}

with open('../models/checkpoints/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Guardado: models/checkpoints/model_metadata.json')
print()
print(json.dumps(metadata, indent=2))

## 9. Test de predict() con datos crudos

> Simula exactamente cómo lo llamará el Integrante 3 desde la app Streamlit.
> La función aplica el pipeline del Integrante 1 internamente.

In [ ]:
from src.models.predict import predict

sample_low_risk = {
    'Age': 45,
    'Sex': 'male',
    'Job': 2,
    'Housing': 'own',
    'Saving accounts': 'rich',
    'Checking account': 'moderate',
    'Credit amount': 2000,
    'Duration': 12,
    'Purpose': 'car'
}

sample_high_risk = {
    'Age': 22,
    'Sex': 'female',
    'Job': 0,
    'Housing': 'free',
    'Saving accounts': 'unknown',
    'Checking account': 'unknown',
    'Credit amount': 15000,
    'Duration': 60,
    'Purpose': 'education'
}

print('=== Esperado: BAJO RIESGO ===')
print(predict(sample_low_risk, model_name='best_model'))

print('\n=== Esperado: ALTO RIESGO ===')
print(predict(sample_high_risk, model_name='best_model'))

---
## Resumen para el equipo

| Archivo | Uso |
|---------|-----|
| `models/checkpoints/best_model.pkl` | Modelo entrenado |
| `models/checkpoints/model_metadata.json` | Métricas, features, umbrales |
| `models/plots/*.png` | Gráficas para PDF/video |

**Para el Integrante 3 (Streamlit):**
```python
from src.models.predict import predict

resultado = predict(input_dict, model_name='best_model')
# resultado['risk_label']          → 'Bajo riesgo' / 'Riesgo medio' / 'Alto riesgo'
# resultado['probability_default'] → % de probabilidad de default
```